In [ ]:
# Install required libraries if they are missing
!pip install -q pypdf langchain-text-splitters langchain-community fastembed faiss-cpu groq google-genai python-dotenv

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import FAISS

from groq import Groq
from google import genai

from dotenv import load_dotenv

import os
import hashlib
import gc
import time


# ============================================================
# 1. LOAD API KEYS
# ============================================================

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

print("Groq API key loaded:", bool(groq_api_key))
print("Gemini API key loaded:", bool(google_api_key))


if not groq_api_key and not google_api_key:
    print("\nNo API key found in .env file.")
    exit()


# ============================================================
# 2. CREATE CLOUD CLIENTS
# ============================================================

groq_client = None
gemini_client = None


if groq_api_key:
    try:
        groq_client = Groq(
            api_key=groq_api_key
        )
    except Exception as e:
        print("Groq client error:")
        print(e)


if google_api_key:
    try:
        gemini_client = genai.Client(
            api_key=google_api_key
        )
    except Exception as e:
        print("Gemini client error:")
        print(e)


# ============================================================
# 3. GET PDF PATH
# ============================================================

pdf_path = input(
    "\nEnter PDF file path: "
).strip()


# Remove accidental PowerShell &
if pdf_path.startswith("&"):
    pdf_path = pdf_path[1:].strip()


# Remove quotes
pdf_path = pdf_path.strip('"').strip("'")


# Check PDF
if not os.path.isfile(pdf_path):
    print("\nPDF file not found.")
    print("Path:", pdf_path)
    exit()


print("\nPDF found!")


# ============================================================
# 4. CREATE UNIQUE CACHE ID
# ============================================================

absolute_path = os.path.abspath(pdf_path)

pdf_id = hashlib.md5(
    absolute_path.encode()
).hexdigest()


cache_root = "pdf_vector_cache"

os.makedirs(
    cache_root,
    exist_ok=True
)


cache_path = os.path.join(
    cache_root,
    pdf_id
)


# ============================================================
# 5. LOAD EMBEDDING MODEL
# ============================================================

print("\nLoading embedding model...")


try:

    embeddings = FastEmbedEmbeddings(
        model_name="BAAI/bge-small-en-v1.5"
    )

except Exception as e:

    print("\nEmbedding model error:")
    print(e)
    exit()


# ============================================================
# 6. CHECK IF PDF WAS ALREADY PROCESSED
# ============================================================

if os.path.exists(cache_path):

    print("\nCached PDF found!")
    print("Loading saved vector database...")


    try:

        vector_store = FAISS.load_local(
            cache_path,
            embeddings,
            allow_dangerous_deserialization=True
        )

        print("PDF cache loaded successfully!")


    except Exception as e:

        print("\nCould not load PDF cache:")
        print(e)
        exit()


# ============================================================
# 7. IF NO CACHE -> PROCESS PDF
# ============================================================

else:

    print("\nFirst time processing this PDF.")
    print("Reading PDF...")


    # --------------------------------------------------------
    # READ PDF
    # --------------------------------------------------------

    try:

        reader = PdfReader(
            pdf_path
        )

    except Exception as e:

        print("\nPDF reading error:")
        print(e)
        exit()


    total_pages = len(reader.pages)

    print(
        "Total pages:",
        total_pages
    )


    # --------------------------------------------------------
    # TEXT SPLITTER
    # --------------------------------------------------------

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )


    chunks = []


    # --------------------------------------------------------
    # EXTRACT EACH PAGE
    # --------------------------------------------------------

    print("\nExtracting text and creating chunks...")


    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        try:

            text = page.extract_text()

        except Exception:

            text = None


        if text and text.strip():

            page_chunks = splitter.create_documents(
                [text],
                metadatas=[
                    {
                        "page": page_number,
                        "source": os.path.basename(
                            pdf_path
                        )
                    }
                ]
            )

            chunks.extend(
                page_chunks
            )


        # Progress
        if page_number % 100 == 0:

            print(
                f"Read {page_number}/"
                f"{total_pages} pages"
            )


    print(
        "\nTotal chunks:",
        len(chunks)
    )


    if not chunks:

        print(
            "No readable text found in PDF."
        )
        exit()


    # ========================================================
    # 8. CREATE EMBEDDINGS IN SMALL BATCHES
    # ========================================================

    print(
        "\nCreating embeddings in small batches..."
    )

    print(
        "This is done to reduce RAM usage."
    )


    # Small batch because laptop has limited memory
    batch_size = 25

    vector_store = None

    total_chunks = len(chunks)


    for start in range(
        0,
        total_chunks,
        batch_size
    ):

        end = min(
            start + batch_size,
            total_chunks
        )


        batch = chunks[
            start:end
        ]


        try:

            # First batch creates FAISS
            if vector_store is None:

                vector_store = FAISS.from_documents(
                    batch,
                    embeddings
                )


            # Remaining batches added to same FAISS
            else:

                vector_store.add_documents(
                    batch
                )


        except Exception as e:

            print(
                f"\nEmbedding failed "
                f"between chunks "
                f"{start + 1}-{end}"
            )

            print(e)

            print(
                "\nTry closing other applications "
                "and run again."
            )

            exit()


        print(
            f"Processed "
            f"{end}/{total_chunks} chunks"
        )


        # Free unused Python memory
        del batch

        gc.collect()


    # ========================================================
    # 9. SAVE PDF VECTOR CACHE
    # ========================================================

    if vector_store is None:

        print(
            "Vector store could not be created."
        )

        exit()


    print(
        "\nAll chunks processed successfully!"
    )


    try:

        vector_store.save_local(
            cache_path
        )

        print(
            "PDF cache saved successfully!"
        )

        print(
            "Next time this PDF will load "
            "much faster."
        )


    except Exception as e:

        print(
            "\nCould not save PDF cache:"
        )

        print(e)


    # Release large list
    del chunks

    gc.collect()


# ============================================================
# 10. CREATE RETRIEVER
# ============================================================

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)


# ============================================================
# 11. PDF READY
# ============================================================

print("\n========================================")
print("       PDF STUDY ASSISTANT READY")
print("========================================")

print(
    "Ask questions from your PDF."
)

print(
    "Type 'exit' to stop."
)


# ============================================================
# 12. QUESTION LOOP
# ============================================================

while True:

    question = input(
        "\nAsk a question from the PDF: "
    ).strip()


    # --------------------------------------------------------
    # EXIT
    # --------------------------------------------------------

    if question.lower() == "exit":

        print(
            "\nExiting PDF Study Assistant..."
        )

        break


    # --------------------------------------------------------
    # EMPTY QUESTION
    # --------------------------------------------------------

    if not question:

        print(
            "Please enter a question."
        )

        continue


    # ========================================================
    # 13. RETRIEVE RELEVANT PDF CONTENT
    # ========================================================

    print(
        "\nSearching PDF..."
    )


    try:

        relevant_docs = retriever.invoke(
            question
        )


    except Exception as e:

        print(
            "\nRetrieval error:"
        )

        print(e)

        continue


    if not relevant_docs:

        print(
            "No relevant information found."
        )

        continue


    # ========================================================
    # 14. CREATE CONTEXT + PAGE NUMBERS
    # ========================================================

    context_parts = []

    pages_found = []


    for doc in relevant_docs:

        page_number = doc.metadata.get(
            "page",
            "Unknown"
        )


        pages_found.append(
            page_number
        )


        context_parts.append(
            f"[PDF Page {page_number}]\n"
            f"{doc.page_content}"
        )


    context = "\n\n".join(
        context_parts
    )


    # ========================================================
    # 15. CREATE RAG PROMPT
    # ========================================================

    prompt = f"""
You are an AI Study Assistant.

The student uploaded a PDF/book.

Answer the student's question ONLY using the
retrieved PDF context provided below.

RULES:

1. Use only the provided PDF context.
2. Do not invent information.
3. Explain concepts clearly and simply.
4. If the student asks you to teach something,
   explain it step-by-step.
5. Preserve important definitions and algorithms.
6. Use examples from the PDF when available.
7. Mention relevant PDF page numbers when useful.
8. If the answer cannot be found in the context,
   say exactly:

"I could not find this information in the PDF."


PDF CONTEXT:

{context}


STUDENT QUESTION:

{question}


ANSWER:
"""


    answer_generated = False


    # ========================================================
    # 16. PRIMARY PROVIDER -> GROQ
    # ========================================================

    if groq_client:

        print(
            "\nTrying Groq..."
        )


        try:

            response = (
                groq_client
                .chat
                .completions
                .create(

                    model="llama-3.3-70b-versatile",

                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],

                    temperature=0.2
                )
            )


            answer = (
                response
                .choices[0]
                .message
                .content
            )


            if answer:

                print(
                    "\n--- FINAL ANSWER ---\n"
                )

                print(answer)


                print(
                    "\nRetrieved PDF pages:",
                    sorted(
                        set(pages_found),
                        key=lambda x: (
                            isinstance(x, str),
                            x
                        )
                    )
                )


                print(
                    "\nAnswered by: Groq"
                )


                answer_generated = True


        except Exception as e:

            print(
                "\nGroq failed:"
            )

            print(e)


    # ========================================================
    # 17. FALLBACK PROVIDER -> GEMINI
    # ========================================================

    if (
        not answer_generated
        and gemini_client
    ):

        print(
            "\nSwitching to Gemini..."
        )


        try:

            response = (
                gemini_client
                .models
                .generate_content(

                    model="gemini-3.8-flash",

                    contents=prompt
                )
            )


            if response.text:

                print(
                    "\n--- FINAL ANSWER ---\n"
                )

                print(
                    response.text
                )


                print(
                    "\nRetrieved PDF pages:",
                    sorted(
                        set(pages_found),
                        key=lambda x: (
                            isinstance(x, str),
                            x
                        )
                    )
                )


                print(
                    "\nAnswered by: Gemini"
                )


                answer_generated = True


        except Exception as e:

            print(
                "\nGemini failed:"
            )

            print(e)


    # ========================================================
    # 18. BOTH CLOUD PROVIDERS FAILED
    # ========================================================

    if not answer_generated:

        print(
            "\nThe PDF retrieval worked, "
            "but both cloud LLM providers failed."
        )

        print(
            "You can ask another question "
            "without processing the PDF again."
        )

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_471/1231132471.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings.fastembed import FastEmbedEmbeddings


Groq API key loaded: False
Gemini API key loaded: False

No API key found in .env file.


KeyboardInterrupt: Interrupted by user

In [ ]:
!pip install -q pypdf langchain-text-splitters langchain-community faiss-cpu fastembed groq google-genai

In [10]:
from google.colab import userdata
import os

groq_api_key = userdata.get("GROQ_API_KEY")
google_api_key = userdata.get("GOOGLE_API_KEY")

print("Groq API key loaded:", bool(groq_api_key))
print("Gemini API key loaded:", bool(google_api_key))

Groq API key loaded: True
Gemini API key loaded: True


In [11]:
from google.colab import files
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import FAISS

from groq import Groq
from google import genai

import gc

# -----------------------------
# LLM clients
# -----------------------------

groq_client = Groq(api_key=groq_api_key)
gemini_client = genai.Client(api_key=google_api_key)

# -----------------------------
# Upload PDF
# -----------------------------

print("Upload your PDF:")

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("\nUploaded:", pdf_path)

# -----------------------------
# Read PDF
# -----------------------------

reader = PdfReader(pdf_path)

print("Total pages:", len(reader.pages))

# -----------------------------
# Text splitter
# -----------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = []

print("\nExtracting PDF text...")

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text()

    if text and text.strip():

        page_chunks = splitter.create_documents(
            [text],
            metadatas=[
                {
                    "page": page_number,
                    "source": pdf_path
                }
            ]
        )

        chunks.extend(page_chunks)

    if page_number % 100 == 0:
        print(
            f"Processed pages: "
            f"{page_number}/{len(reader.pages)}"
        )

print("\nTotal chunks:", len(chunks))

# -----------------------------
# Embeddings
# -----------------------------

print("\nLoading embedding model...")

embeddings = FastEmbedEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# -----------------------------
# FAISS in batches
# -----------------------------

print("\nCreating vector database...")

batch_size = 50
vector_store = None

for start in range(0, len(chunks), batch_size):

    end = min(start + batch_size, len(chunks))

    batch = chunks[start:end]

    if vector_store is None:

        vector_store = FAISS.from_documents(
            batch,
            embeddings
        )

    else:

        vector_store.add_documents(
            batch
        )

    print(
        f"Embedded chunks: {end}/{len(chunks)}"
    )

    del batch
    gc.collect()

print("\nVector database created successfully!")

# -----------------------------
# Retriever
# -----------------------------

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("\n================================")
print("PDF RAG READY")
print("================================")

/tmp/ipykernel_970/1245978828.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings.fastembed import FastEmbedEmbeddings


Upload your PDF:


Saving Introduction.to_.Algorithms.4th.Leiserson.Stein_.Rivest.Cormen.MIT_.Press.pdf to Introduction.to_.Algorithms.4th.Leiserson.Stein_.Rivest.Cormen.MIT_.Press (1).pdf

Uploaded: Introduction.to_.Algorithms.4th.Leiserson.Stein_.Rivest.Cormen.MIT_.Press (1).pdf
Total pages: 1312

Extracting PDF text...
Processed pages: 100/1312
Processed pages: 200/1312
Processed pages: 300/1312
Processed pages: 400/1312
Processed pages: 500/1312
Processed pages: 600/1312
Processed pages: 700/1312
Processed pages: 800/1312
Processed pages: 900/1312
Processed pages: 1000/1312
Processed pages: 1100/1312
Processed pages: 1200/1312
Processed pages: 1300/1312

Total chunks: 3652

Loading embedding model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


Creating vector database...
Embedded chunks: 50/3652
Embedded chunks: 100/3652
Embedded chunks: 150/3652
Embedded chunks: 200/3652
Embedded chunks: 250/3652
Embedded chunks: 300/3652
Embedded chunks: 350/3652
Embedded chunks: 400/3652
Embedded chunks: 450/3652
Embedded chunks: 500/3652
Embedded chunks: 550/3652
Embedded chunks: 600/3652
Embedded chunks: 650/3652
Embedded chunks: 700/3652
Embedded chunks: 750/3652
Embedded chunks: 800/3652
Embedded chunks: 850/3652
Embedded chunks: 900/3652
Embedded chunks: 950/3652
Embedded chunks: 1000/3652
Embedded chunks: 1050/3652
Embedded chunks: 1100/3652
Embedded chunks: 1150/3652
Embedded chunks: 1200/3652
Embedded chunks: 1250/3652
Embedded chunks: 1300/3652
Embedded chunks: 1350/3652
Embedded chunks: 1400/3652
Embedded chunks: 1450/3652
Embedded chunks: 1500/3652
Embedded chunks: 1550/3652
Embedded chunks: 1600/3652
Embedded chunks: 1650/3652
Embedded chunks: 1700/3652
Embedded chunks: 1750/3652
Embedded chunks: 1800/3652
Embedded chunks: 18

In [15]:
# ============================================================
# CELL 4 - PDF RAG QUESTION ANSWERING
# Primary: GROQ
# Fallback: GEMINI
# ============================================================

while True:

    # --------------------------------------------------------
    # 1. USER QUESTION
    # --------------------------------------------------------

    question = input(
        "\nAsk a question from the PDF (or type 'exit'): "
    ).strip()


    # Exit
    if question.lower() == "exit":
        print("\nStudy Assistant stopped.")
        break


    # Empty question
    if not question:
        print("Please enter a question.")
        continue


    # --------------------------------------------------------
    # 2. RETRIEVE RELEVANT PDF CONTENT
    # --------------------------------------------------------

    print("\nSearching PDF...")

    try:

        relevant_docs = retriever.invoke(question)

    except Exception as e:

        print("\nRetrieval Error:")
        print(e)
        continue


    if not relevant_docs:

        print(
            "\nI could not find relevant information "
            "in the PDF."
        )

        continue


    # --------------------------------------------------------
    # 3. CREATE CONTEXT + PAGE NUMBERS
    # --------------------------------------------------------

    context_parts = []
    pages_found = []


    for doc in relevant_docs:

        page = doc.metadata.get(
            "page",
            "Unknown"
        )

        pages_found.append(page)

        context_parts.append(
            f"[PDF Page {page}]\n"
            f"{doc.page_content}"
        )


    context = "\n\n".join(context_parts)


    # --------------------------------------------------------
    # 4. CREATE RAG PROMPT
    # --------------------------------------------------------

    prompt = f"""
You are an AI Study Assistant.

The student uploaded a PDF/book.

Answer the student's question ONLY using the
retrieved PDF context given below.

RULES:

1. Use only the provided PDF context.
2. Do not invent information.
3. Explain concepts in simple language.
4. If the student asks you to teach something,
   explain it step-by-step.
5. Preserve important definitions, algorithms,
   formulas, pseudocode, and examples when available.
6. Mention relevant PDF page numbers when useful.
7. If the answer is not available in the context,
   say exactly:

"I could not find this information in the PDF."


PDF CONTEXT:

{context}


STUDENT QUESTION:

{question}


ANSWER:
"""


    # --------------------------------------------------------
    # 5. ANSWER STATUS
    # --------------------------------------------------------

    answer_generated = False


    # ========================================================
    # 6. PRIMARY PROVIDER - GROQ
    # ========================================================

    if groq_client:

        print("\nTrying Groq...")


        try:

            response = (
                groq_client
                .chat
                .completions
                .create(

                    # Current Groq model
                    model="openai/gpt-oss-120b",

                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],

                    temperature=0.2
                )
            )


            answer = (
                response
                .choices[0]
                .message
                .content
            )


            # ------------------------------------------------
            # GROQ SUCCESS
            # ------------------------------------------------

            if answer:

                print(
                    "\n=============================="
                )

                print(
                    "FINAL ANSWER"
                )

                print(
                    "==============================\n"
                )

                print(answer)


                print(
                    "\nRetrieved PDF pages:",
                    sorted(set(pages_found))
                )


                print(
                    "\nAnswered by: Groq"
                )


                answer_generated = True


        # ----------------------------------------------------
        # GROQ FAILED
        # ----------------------------------------------------

        except Exception as e:

            print(
                "\n========== GROQ ERROR =========="
            )

            print(e)

            print(
                "================================"
            )

            print(
                "\nGroq failed."
            )

            print(
                "Switching to Gemini fallback..."
            )


    # ========================================================
    # 7. FALLBACK PROVIDER - GEMINI
    # ========================================================

    if (
        not answer_generated
        and gemini_client
    ):

        print(
            "\nTrying Gemini fallback..."
        )


        try:

            response = (
                gemini_client
                .models
                .generate_content(

                    model="gemini-3.8-flash",

                    contents=prompt
                )
            )


            # ------------------------------------------------
            # GEMINI SUCCESS
            # ------------------------------------------------

            if response.text:

                print(
                    "\n=============================="
                )

                print(
                    "FINAL ANSWER"
                )

                print(
                    "==============================\n"
                )

                print(
                    response.text
                )


                print(
                    "\nRetrieved PDF pages:",
                    sorted(set(pages_found))
                )


                print(
                    "\nAnswered by: Gemini"
                )


                answer_generated = True


        # ----------------------------------------------------
        # GEMINI FAILED
        # ----------------------------------------------------

        except Exception as e:

            print(
                "\n========== GEMINI ERROR =========="
            )

            print(e)

            print(
                "=================================="
            )


    # ========================================================
    # 8. BOTH PROVIDERS FAILED
    # ========================================================

    if not answer_generated:

        print(
            "\n=================================="
        )

        print(
            "ALL CLOUD PROVIDERS FAILED"
        )

        print(
            "=================================="
        )


        print(
            "\nYour PDF retrieval is working."
        )

        print(
            "Only the cloud LLM providers "
            "are currently unavailable."
        )

        print(
            "\nYou can ask another question."
        )

        print(
            "You DO NOT need to upload or "
            "process the PDF again."
        )


Ask a question from the PDF (or type 'exit'): Teach me insertion sort from this book in simple language.

Searching PDF...

Trying Groq...

FINAL ANSWER

**Insertion Sort – a simple “hand‑of‑cards” sorting method**  

*Source: pages 39‑41, 40, 52 of the PDF.*

---

### 1. What the algorithm does  
Insertion sort builds a sorted part of the array one element at a time, just like you would sort a hand of playing cards:

1. Start with the first card (the first array element) in your left hand – this is already “sorted”.
2. Pick up the next card (the next element) and slide it leftward, comparing it with the cards already in your hand, until you find the right spot.
3. Insert the card there.  
4. Repeat for every remaining card.

When the process finishes, the whole hand (the whole array) is sorted.  
(See the intuitive description on page 40 and the picture on page 41.)

---

### 2. Pseudocode (the book’s version)

```
INSERTION‑SORT(A)
1  for i = 2 to n
2      key = A[i]                

KeyboardInterrupt: Interrupted by user

In [16]:
!git init
!git status

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	Introduction.to_.Algorithms.4th.Leiserson.Stein_.Rivest.Cormen.MIT_.Press (1).pdf
	Introduction.to_.Algorithms.4th.Leiserson.Stein_.Rivest.Cormen.MIT_.Press.pdf
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)


In [17]:
%%writefile .gitignore
.env
*.pdf
__pycache__/
.ipynb_checkpoints/
vector_cache/
pdf_vector_cache/

Writing .gitignore


In [18]:
!git clone https://github.com/ashishyaduveer791-blip/LLM_Model.git

Cloning into 'LLM_Model'...


In [19]:
%cd LLM_Model

/content/LLM_Model


In [20]:
!git status

On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)
